# 03 · Cross-Encoders & Reranking
Fast retrieval (bi-encoder / vector search) is good but imperfect. A **cross-encoder** re-scores the top candidates much more accurately — the standard **retrieve-then-rerank** pattern.

## 1. Bi-encoder vs cross-encoder
```
  BI-ENCODER (retrieval)                 CROSS-ENCODER (reranking)
  ----------------------                 -------------------------
  embed query  --> [vec_q]               feed (query, doc) TOGETHER
  embed doc    --> [vec_d]  (separately)   into ONE model -> a relevance score
  score = cosine(vec_q, vec_d)           model sees them jointly -> far more accurate

  FAST: docs embedded once, ahead of     SLOW: must run the model for EVERY
        time; query is one embed.              (query, doc) pair at query time.
```
Bi-encoders can index millions offline. Cross-encoders are too slow for that — so you use them only on the ~top 20 candidates a bi-encoder already found.

## 2. The retrieve-then-rerank pipeline
```
  query --> [ bi-encoder retrieval ] --> top 50 candidates
                                             |
                                             v
                    [ cross-encoder reranker ] --> re-ordered top 5 --> generator
```

In [ ]:
# Conceptual demo with a MOCK cross-encoder (offline). A real one is e.g.
#   from sentence_transformers import CrossEncoder
#   ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
#   scores = ce.predict([(query, doc) for doc in candidates])
#
# Our mock scores a (query, doc) pair by weighted word overlap that also rewards
# ORDER/proximity a bit — standing in for the cross-encoder's joint reasoning.
import re
def toks(s): return re.findall(r"[a-z]+", s.lower())
def mock_cross_encoder(query, doc):
    q, d = toks(query), toks(doc)
    overlap = len(set(q) & set(d)) / (len(set(q)) or 1)
    # small bonus if query words appear close together in the doc
    bonus = 0.2 if all(w in d for w in q[:2]) else 0.0
    return round(overlap + bonus, 3)

query = "waterproof jacket for winter hiking"
candidates = [
    "this jacket is waterproof and windproof, ideal for winter hiking",   # best
    "our running shoes are lightweight and breathable",
    "the wool sweater is warm and machine washable",
    "a waterproof phone case for outdoor use",                            # partial
]
scored = sorted(((mock_cross_encoder(query,c), c) for c in candidates), reverse=True)
print("Reranked:")
for s, c in scored: print(f"  {s}  {c}")

**Observe:** the cross-encoder pushes the exactly-on-topic jacket to the top and demotes the partial 'waterproof phone case', because it judges the pair *jointly* rather than as two separate vectors. In production this typically lifts precision@k noticeably over vector search alone.

**Cost trade-off:** rerank only a small candidate set (e.g. top 20–50). Reranking the whole corpus would be far too slow — that's the bi-encoder's job.